In [1]:
import os
os.environ["PYTHONNET_RUNTIME"] = "coreclr"
os.environ["DOTNET_SYSTEM_DRAWING_USE_GDIPLUS"] = "1"

import clr
import numpy as np
import time
from datetime import timedelta
from pythonnet import load
from System import Environment, Array, String

# Load .NET Core runtime
load("coreclr")

# Try to import System.IO; fallback to os.chdir if it fails
try:
    from System.IO import Directory, Path, File
    use_dotnet_dir = True
except Exception as e:
    print(f"Note: System.IO import failed ({e}), using os.chdir instead")
    import os
    use_dotnet_dir = False

# Define DWSIM path
dwsimpath = "/usr/local/lib/dwsim/"

# Set working directory
if use_dotnet_dir:
    Directory.SetCurrentDirectory(dwsimpath)
else:
    os.chdir(dwsimpath)

# Add DWSIM assemblies
clr.AddReference(dwsimpath + "CapeOpen.dll")
clr.AddReference(dwsimpath + "DWSIM.Automation.dll")
clr.AddReference(dwsimpath + "DWSIM.Interfaces.dll")
clr.AddReference(dwsimpath + "DWSIM.GlobalSettings.dll")
clr.AddReference(dwsimpath + "DWSIM.SharedClasses.dll")
clr.AddReference(dwsimpath + "DWSIM.Thermodynamics.dll")
clr.AddReference(dwsimpath + "DWSIM.UnitOperations.dll")
clr.AddReference(dwsimpath + "DWSIM.Inspector.dll")
clr.AddReference(dwsimpath + "System.Buffers.dll")
clr.AddReference(dwsimpath + "DWSIM.Thermodynamics.ThermoC.dll")

# Now import DWSIM types
from DWSIM.Interfaces.Enums.GraphicObjects import ObjectType
from DWSIM.Thermodynamics import Streams, PropertyPackages
from DWSIM.UnitOperations import UnitOperations
from DWSIM.Automation import Automation3
from DWSIM.GlobalSettings import Settings

print("DWSIM imports successful!")

DWSIM imports successful!


In [2]:
# Create an instance of the Automation3 class from the DWSIM.Automation module
# This class provides methods for automating tasks in DWSIM, such as creating and manipulating flowsheets
interf = Automation3()

In [3]:
# Creates a flowsheet
sim = interf.CreateFlowsheet()

In [4]:
# Add Compounds
sim.AddCompound("Carbon monoxide")
sim.AddCompound("Water")
sim.AddCompound("Carbon dioxide")
sim.AddCompound("Hydrogen")

In [5]:
steam = sim.AddFlowsheetObject('Material Stream','steam')
hydrogen = sim.AddFlowsheetObject('Material Stream','hydrogen')
flue_gas = sim.AddFlowsheetObject('Material Stream','flue_gas')
feed = sim.AddFlowsheetObject('Material Stream','feed')
syngas = sim.AddFlowsheetObject('Material Stream','syngas')
residue = sim.AddFlowsheetObject('Material Stream', 'residue')
air_fuel_mixer = sim.AddFlowsheetObject('Stream Mixer','air_fuel_mixer')
reactor = sim.AddFlowsheetObject('Equilibrium Reactor','reactor')

In [6]:
steam = steam.GetAsObject()
hydrogen = hydrogen.GetAsObject()
syngas = syngas.GetAsObject()
flue_gas = flue_gas.GetAsObject()
feed = feed.GetAsObject()
residue = residue.GetAsObject()
air_fuel_mixer = air_fuel_mixer.GetAsObject()
reactor = reactor.GetAsObject()

In [7]:
sim.ConnectObjects(steam.GraphicObject, air_fuel_mixer.GraphicObject, -1, -1)
sim.ConnectObjects(hydrogen.GraphicObject, air_fuel_mixer.GraphicObject, -1, -1)
sim.ConnectObjects(flue_gas.GraphicObject, air_fuel_mixer.GraphicObject, -1, -1)
sim.ConnectObjects(air_fuel_mixer.GraphicObject, feed.GraphicObject, -1, -1)
sim.ConnectObjects(feed.GraphicObject, reactor.GraphicObject, -1, -1)
sim.ConnectObjects(reactor.GraphicObject, syngas.GraphicObject, -1, -1)
sim.ConnectObjects(reactor.GraphicObject, residue.GraphicObject, -1, -1)

In [8]:
sim.AutoLayout()

In [9]:
flue_gas.SetOverallComposition(Array[float]([0.5, 0.0, 0.5, 0.0])) # Flue gas composition (CO and CO2)
flue_gas.SetTemperature(300.0) # K
flue_gas.SetPressure(101325.0) # Pa
flue_gas.SetMassFlow(1.0) # kg/s

steam.SetOverallComposition(Array[float]([0.0, 1.0, 0.0, 0.0])) # Steam composition (H2O and N2)
steam.SetTemperature(400.0) # K
steam.SetPressure(101325.0) # Pa
steam.SetMassFlow(5.0) # kg/s

hydrogen.SetOverallComposition(Array[float]([0.0, 0.0, 0.0, 1.0])) # Hydrogen composition (H2)  
hydrogen.SetTemperature(300.0) # K
hydrogen.SetPressure(101325.0) # Pa
hydrogen.SetMassFlow(0.5) # kg/s

'hydrogen: mass flow set to 0.5 kg/s'

In [10]:
# property package
Thermo_Package_PR = sim.CreateAndAddPropertyPackage("Peng-Robinson (PR)")
Thermo_Package_ST = sim.CreateAndAddPropertyPackage("Steam Tables (IAPWS-IF97)")

In [11]:
steam.SetPropertyPackage(Thermo_Package_ST)

In [12]:
# Creating the conversion reaction for CH4 combustion
from System.Collections.Generic import Dictionary
from System import String, Double

# Stoichiometric coefficients (positive for products, negative for reactants)
stoich = Dictionary[String, Double]()
stoich.Add("Carbon monoxide", -1.0)   # CO
stoich.Add("Water", -1.0)             # H2O
stoich.Add("Carbon dioxide", 1.0)      # CO2
stoich.Add("Hydrogen", 1.0)           # H2
# Base compound (not strictly needed for equilibrium, but required)
base_compound = "Carbon monoxide"

# Reaction phase – use "Mixture" for both vapour and liquid
reaction_phase = "Vapor"

# Basis – commonly "Fugacity" for gas‑phase equilibria, or "Activity" for liquids
basis = "Fugacity"

# Units – the unit set for the basis; often "SI" or an empty string.
# For fugacity basis, units may be ignored. We'll use an empty string.
units = ""

# Temperature approach (K) – 0 means no approach
Tapproach = 0.0

# ln(Keq) expression as a function of T (in K).
# If left empty, DWSIM calculates Keq from Gibbs energies of formation.
# Here we provide a simple constant (Keq = 10) as an example.
lnKeq_expression = ""   # empty string means DWSIM will calculate Keq internally

# Create the reaction
eq_reaction = sim.CreateEquilibriumReaction(
    "WaterGasShift",                     # name
    "Water‑gas shift equilibrium",        # description
    stoich,                               # stoichiometry
    base_compound,                        # base compound
    reaction_phase,                       # phase
    basis,                                 # basis
    units,                                 # units
    Tapproach,                             # temperature approach
    lnKeq_expression                      # ln(Keq) expression
)

# Add the reaction to the flowsheet
sim.AddReaction(eq_reaction)

# ------------------------------------------------------------
# 3. Create a reaction set and add the equilibrium reaction
# ------------------------------------------------------------
reaction_set = sim.CreateReactionSet("WGS Set", "Contains water‑gas shift reaction")
sim.AddReactionSet(reaction_set)

# Add reaction to the set (enabled, rank 0)
sim.AddReactionToSet(eq_reaction.Name, reaction_set.Name, True, 0)

# Setting reaction set to the conversion reactor
from DWSIM.UnitOperations.Reactors import OperationMode
reactor.ReactorOperationMode = OperationMode.Isothermic
reactor.DeltaP = 0.5
reactor.ReactionSetID = reaction_set.Name
reactor.ReactionSet = reaction_set

In [13]:
# request a calculation
errors = interf.CalculateFlowsheet4(sim)
list(errors)

[]

In [14]:
# save file

fileNameToSave = Path.Combine(Environment.GetFolderPath(Environment.SpecialFolder.Desktop), "/workspace/00 FlowSheet Automation/13 Equilibrium Reactor Automation/13 Equilibrium Reactor Automation.dwxmz")

interf.SaveFlowsheet(sim, fileNameToSave, True)